# ViT-LENS: Notebook 02 — Vision Transformer Activation Extraction
**Research Goal**: Load pretrained frozen ViT-B/16, attach PyTorch forward hooks to target intermediate Transformer Block (Block 7), and stream patch activations into manageable chunk files (`chunk_*.pt`).


In [ ]:
# Step 1: Initialize Frozen Pretrained ViT-B/16
import torch
from src.utils import get_device, load_config
from src.vit_model import ViTModelWrapper
from src.dataset import build_dataloaders
from src.activation_extractor import ActivationExtractor

device = get_device('auto')
config = load_config('config.yaml')

vit = ViTModelWrapper(
    model_name=config['vit']['model_name'],
    pretrained=True,
    target_layer_idx=config['vit']['target_layer_idx'],
    device=device
)
print(f'Loaded ViT model ({vit.model_type}) with target hook on Block #{vit.target_layer_idx}')


In [ ]:
# Step 2: Verify activation shapes on test batch
dummy_x = torch.randn(2, 3, 224, 224).to(device)
logits, probs, patch_acts, cls_acts = vit(dummy_x)

print(f'Logits shape: {logits.shape} (Expected [2, 1000])')
print(f'Patch activations shape: {patch_acts.shape} (Expected [2, 196, 768])')
print(f'CLS token activation shape: {cls_acts.shape} (Expected [2, 1, 768])')
assert patch_acts.shape == (2, 196, 768), 'Shape mismatch on patch tokens!'


In [ ]:
# Step 3: Run chunked extraction pipeline over dataset
data_dir = config['dataset']['data_dir'] + '/prototype'
train_loader, _, _ = build_dataloaders(data_dir, batch_size=16, val_split=0.0, num_workers=0)

extractor = ActivationExtractor(
    vit_wrapper=vit,
    output_dir=config['extraction']['output_dir'],
    chunk_size=config['extraction']['chunk_size'],
    exclude_cls=not config['vit']['include_cls_token']
)

saved_chunks = extractor.extract_and_save(train_loader, max_images=config['dataset']['max_samples'])
print(f'Successfully extracted and stored {len(saved_chunks)} activation chunks.')
